# Why Spark?

---
## Before we start: working in a notebook

A notebook is an ordered collection of **cells**. Markdown cells contain formatted explanation; code cells contain Python that we can run.

Run the current cell with the play button or `Shift+Enter`. Code-cell output appears below the cell. Run this lesson from top to bottom: variables created in one code cell are available to later cells while the session is active.

In Microsoft Fabric, the `spark` session is already available for the Spark code we will use later. Run the practice cell below, then continue through the notebook in order.

In [ ]:
print("Hello, PySpark!")

--- 
#### Why does Spark exist? 
Spark lets us process data that is too large, too slow, or too operationally complex for a single machine. It divides data into **partitions** and distributes work across a cluster.

- It supports data engineering, SQL analytics, streaming, and machine-learning feature preparation with Python, SQL, R, Scala, and Java.
- Each executor can process separate partitions at the same time, so suitable workloads scale horizontally.
- Spark does **not** keep all data in memory automatically. Caching is an explicit choice, and Spark can spill data to disk when necessary.



---
#### Why have Microsoft built their end-to-end data analytics platform around it? 
- The data world have converged on Spark. It is used by thousands of companies, including 80% of the Fortune 500. 
- Spark is relatively simple to learn and modular.  

<img src='https://techcrunch.com/wp-content/uploads/2023/05/Microsoft-Fabric.jpg' width='800px'>





---
#### When should we use Spark in Fabric? 
- A workload needs distributed processing: the data is too large for one machine, or its processing time is no longer acceptable.
- We need repeatable, programmatic data ingestion and transformation pipelines.
- We need Spark SQL, streaming, or distributed feature engineering.
- Text or image data needs large-scale ingestion or preparation (although a specialised library may be better for training a particular model).

Use a local tool such as pandas, Polars, or a Warehouse/SQL endpoint when the data fits comfortably on one machine and that is the simpler solution.



---
### Components

<img src='https://www.oreilly.com/api/v2/epubs/9781492050032/files/assets/lesp_0103.png'>

Read the diagram as: the **driver** creates and coordinates a plan; **executors** run the plan's tasks; each task works on one **partition** of the data.

---
### The Spark execution model

A Spark program has three useful levels to keep separate:

1. The **driver** is where our notebook code runs. It builds a plan and asks the cluster to execute it.
2. **Executors** are worker processes that perform the work.
3. Data is split into **partitions**. A task usually processes one partition, allowing many tasks to run concurrently.

More partitions are not always better: very small partitions create scheduling overhead, while very large ones reduce parallelism or may not fit in memory.

---
### Lazy evaluation: plans first, work later

Most DataFrame methods are **transformations**: `select`, `filter`, `withColumn`, and `groupBy` describe *what* we want, but do not immediately scan the data. Spark can combine and optimise these steps into a plan.

An **action** such as `show()`, `count()`, `collect()`, or `write` triggers execution. `collect()` brings all results to the driver, so use it only when the result is safely small.

In [ ]:
from pyspark.sql import functions as F

numbers = spark.range(1_000_000).repartition(8)
summary = (
    numbers
    .withColumn("bucket", F.col("id") % 10)
    .filter(F.col("id") > 100)
    .groupBy("bucket")
    .count()
)

summary.explain()  # Inspect the plan; no result has been returned yet.
summary.show()     # Action: Spark now executes the plan.

---
### Data movement, caching, and resilience

- Operations such as `filter` can normally work independently on each partition.
- `groupBy`, many joins, `distinct`, and sorting often require a **shuffle**: moving records between executors so matching keys meet. Shuffles are expensive, so they are a key place to look when a job is slow.
- Use `cache()` / `persist()` only when an expensive DataFrame is reused; release it with `unpersist()` when finished.
- Spark records the lineage of transformations. If a partition is lost, it can usually recompute that partition instead of restarting the entire job.

---
### Quick check

Before running the example, predict: which line triggers execution? Which operation is likely to cause a shuffle? Why would `summary.collect()` be safe here but risky for a result with millions of rows?